# PointBigBird-JEPA — training **(input-pos-only ablation)**

This notebook is identical to `PBB_JEPA_train.ipynb` except that the
**per-layer positional re-injection is disabled** (`cfg.reinject_pos = False`).
The positional embedding `γ(coord)` is added *only* by the tokenizer at the
input; the encoder blocks see no further positional residual.

Use this notebook to ablate the per-layer pos-residual idea and compare
against the default training notebook (which re-injects pos at every block).

Checkpoint dir is `./checkpoints_pbb_input_pos_only` so it won't clash
with the default run.


## 1. Setup & config

In [ ]:
import os, sys, math, time, copy, ssl
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

ssl._create_default_https_context = ssl._create_unverified_context

from pbb import (
    PBBConfig, PBBEncoder, PBBPredictor,
    build_loaders, orderings_from_batch,
    TargetCenter, ema_update, make_momentum_schedule,
    gather_target_features, jepa_loss, diag_dict, fmt_diag,
    save_atomic, ensure_dir, short_params, count_params,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

cfg = PBBConfig()

# ---- Override what you want to change here ----
cfg.epochs         = 100
cfg.batch_size     = 64
cfg.lr             = 2e-4
cfg.probe_interval = 20
cfg.probe_epochs   = 3
cfg.log_every      = 50
cfg.ckpt_dir       = "./checkpoints_pbb_input_pos_only"   # ablation: separate dir
cfg.reinject_pos   = False                                # *** ablation toggle ***

ensure_dir(cfg.ckpt_dir)
print(f"DEVICE = {DEVICE}")
print(f"K_POOL={cfg.k_pool}  K_CTX={cfg.k_ctx}  N_TGT={cfg.n_tgt}  "
      f"D_model={cfg.d_model}  block_size={cfg.block_size}")
print(f"epochs={cfg.epochs}  batch={cfg.batch_size}  lr={cfg.lr}  "
      f"probe every {cfg.probe_interval} ep")


## 2. Data loaders

In [ ]:
train_loader, train_eval_loader, test_loader = build_loaders(cfg, num_workers=2)
print(f"train       : {len(train_loader.dataset)} samples, {len(train_loader)} batches")
print(f"train_eval  : {len(train_eval_loader.dataset)} samples (K_HALF context for probe)")
print(f"test        : {len(test_loader.dataset)} samples (K_HALF context for probe)")

# Peek at one batch
batch = next(iter(train_loader))
print("\nbatch tensor shapes:")
for k, v in batch.items():
    if hasattr(v, 'shape'):
        print(f"  {k:24s} {tuple(v.shape)}  {v.dtype}")


## 3. Models

In [ ]:
context_encoder = PBBEncoder(
    d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
    n_heads=cfg.n_heads, dim_head=cfg.dim_head,
    block_size=cfg.block_size, window=cfg.window,
    n_random=cfg.n_random, n_global=cfg.n_global,
    ffn_mult=cfg.ffn_mult,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    serial_orders=cfg.serial_orders,
    reinject_pos=cfg.reinject_pos,
).to(DEVICE)

target_encoder = copy.deepcopy(context_encoder).to(DEVICE)
for q in target_encoder.parameters():
    q.requires_grad_(False)

predictor = PBBPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult,
).to(DEVICE)

target_center = TargetCenter(cfg.d_model, momentum=cfg.center_momentum).to(DEVICE)

print(f"context_encoder : {short_params(context_encoder)}  ({count_params(context_encoder):,} params)")
print(f"predictor       : {short_params(predictor)}  ({count_params(predictor):,} params)")
print(f"target_encoder  : {short_params(target_encoder)}  (frozen, EMA-updated)")


## 4. Optimizer + schedules

In [ ]:
trainable = list(context_encoder.parameters()) + list(predictor.parameters())
optimizer = AdamW(trainable, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)
print(f"total_steps = {total_steps}  ({cfg.epochs} epochs × {len(train_loader)} batches)")
print(f"EMA momentum: {cfg.ema_start} → {cfg.ema_end}")


## 5. Resume from checkpoint (if any)

In [ ]:
PBB_LAST = os.path.join(cfg.ckpt_dir, "pbb_last.pt")
PBB_BEST = os.path.join(cfg.ckpt_dir, "pbb_best.pt")

# ---- toggle: resume from pbb_last.pt if it exists ----
# Set RESUME=False to start fresh, ignoring any existing checkpoints.
RESUME = True

start_epoch = 1
best_loss   = float("inf")
global_step = 0
history     = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
m           = cfg.ema_start

if RESUME and os.path.exists(PBB_LAST):
    try:
        s = torch.load(PBB_LAST, map_location=DEVICE, weights_only=False)
        context_encoder.load_state_dict(s["context_encoder"])
        if "target_encoder" in s:
            target_encoder.load_state_dict(s["target_encoder"])
        if "predictor" in s:
            predictor.load_state_dict(s["predictor"])
        if "center" in s:
            target_center.load_state_dict(s["center"])
        if "opt" in s:
            optimizer.load_state_dict(s["opt"])
        if "sched" in s:
            scheduler.load_state_dict(s["sched"])
        if "history" in s:
            history = s["history"]
        if "global_step" in s:
            global_step = s["global_step"]
        if "best_loss" in s:
            best_loss = s["best_loss"]
        start_epoch = s["epoch"] + 1
        # Advance momentum schedule
        for _ in range(global_step):
            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
        print(f"Resumed from epoch {s['epoch']}  step {global_step}  best_loss={best_loss:.4f}")
    except Exception as e:
        print(f"Could not resume ({e}); starting fresh.")
else:
    if not RESUME:
        print(f"RESUME=False — starting fresh; existing {PBB_LAST} (if any) will be overwritten")
    else:
        print(f"Starting fresh — no {PBB_LAST}")


## 6. Embedded linear probe (uses `train_eval_loader` for K_HALF input)

In [ ]:
# v8 probe uses train_eval_loader (K_HALF=205 context — matches the test contract)
# so its train/eval distributions agree. Lives in pbb.probe and is shared with train.py.
from pbb import quick_probe

def probe_now(num_epochs=cfg.probe_epochs, lr=1e-3):
    return quick_probe(
        context_encoder, train_eval_loader, test_loader,
        d_model=cfg.d_model, n_classes=10,
        num_epochs=num_epochs, lr=lr, weight_decay=1e-4,
        device=DEVICE,
    )

# Quick smoke check: ensure the encoder forward through one batch works.
_b = next(iter(train_eval_loader))
print("probe-mode batch keys :", [k for k in _b if "perm" not in k and "inv" not in k])
print("probe-mode ctx shape  :", tuple(_b["ctx_pixels"].shape))


## 7. Training loop

In [ ]:
def move_orderings(ords, device):
    """Move a {name: {perm, inverse}} orderings dict onto `device`."""
    return {k: {kk: vv.to(device) for kk, vv in v.items()} for k, v in ords.items()}


epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            ctx_p  = batch["ctx_pixels"].to(DEVICE)
            ctx_c  = batch["ctx_coords"].to(DEVICE)
            pool_p = batch["pool_pixels"].to(DEVICE)
            pool_c = batch["pool_coords"].to(DEVICE)
            tgt_c  = batch["tgt_coords"].to(DEVICE)
            tgt_pp = batch["tgt_pool_pos"].to(DEVICE)
            ctx_o  = move_orderings(orderings_from_batch(batch, "ctx"),  DEVICE)
            pool_o = move_orderings(orderings_from_batch(batch, "pool"), DEVICE)

            with torch.no_grad():
                g_tgt = target_encoder(pool_p, pool_c, pool_o)
                h_tgt_raw = gather_target_features(g_tgt, tgt_pp)
                target_center.update(h_tgt_raw)
                h_tgt = F.layer_norm(target_center(h_tgt_raw), (h_tgt_raw.size(-1),))

            g_ctx  = context_encoder(ctx_p, ctx_c, ctx_o)
            h_pred = predictor(g_ctx, tgt_c)

            loss = jepa_loss(h_pred, h_tgt)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            scheduler.step()

            try:    m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(target_encoder, context_encoder, m)

            global_step += 1
            epoch_loss += loss.item(); steps += 1

            if global_step % cfg.log_every == 0:
                d = diag_dict(loss, h_pred, h_tgt, g_ctx, target_center)
                print(fmt_diag(d, global_step, epoch, scheduler.get_last_lr()[0], m))
                history["diag_steps"].append(global_step)
                history["diag_log"].append(d)

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved:
            best_loss = avg
            save_atomic({
                "epoch": epoch, "cfg": cfg.__dict__,
                "context_encoder": context_encoder.state_dict(),
                "target_encoder":  target_encoder.state_dict(),
                "predictor":       predictor.state_dict(),
                "center":          target_center.state_dict(),
                "opt":             optimizer.state_dict(),
                "sched":           scheduler.state_dict(),
                "global_step":     global_step,
                "best_loss":       best_loss,
                "history":         history,
            }, PBB_BEST)
        save_atomic({
            "epoch": epoch,
            "context_encoder": context_encoder.state_dict(),
            "target_encoder":  target_encoder.state_dict(),
            "predictor":       predictor.state_dict(),
            "center":          target_center.state_dict(),
            "opt":             optimizer.state_dict(),
            "sched":           scheduler.state_dict(),
            "global_step":     global_step,
            "best_loss":       best_loss,
            "history":         history,
        }, PBB_LAST)

        tag = "  ★ new best (saved pbb_best.pt)" if improved else ""
        print(f"=== ep {epoch:03d}/{cfg.epochs}  avg_loss={avg:.4f}  "
              f"m={m:.4f}  {time.time()-t0:.1f}s{tag}")

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_probe = time.time()
            print(f"  → Running {cfg.probe_epochs}-epoch linear probe at epoch {epoch}...")
            acc = probe_now()
            history.setdefault("probe_accs", []).append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc(K_HALF) = {acc:.4f}  ({time.time()-t_probe:.1f}s)")

    print("\nDone.")

except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints: {PBB_LAST}, {PBB_BEST}.")


## 8. Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# (a) per-epoch JEPA loss
if history["loss"]:
    ax = axes[0]
    ax.plot(range(1, len(history["loss"])+1), history["loss"], color='C0', lw=1.5)
    ax.set_xlabel("epoch"); ax.set_ylabel("avg smooth-L1")
    ax.set_yscale("log")
    ax.set_title(f"JEPA loss (best = {min(history['loss']):.4f})")
    ax.grid(alpha=0.3)

# (b) per-step diagnostics
if history["diag_log"]:
    ax = axes[1]
    steps = history["diag_steps"]
    cos_mean = [d["cos_mean"] for d in history["diag_log"]]
    ax.plot(steps, cos_mean, color='C2', lw=1.2, label='cos(h_pred, h_tgt)')
    ax.set_xlabel("step"); ax.set_ylabel("cosine")
    ax.set_ylim(-0.1, 1.05)
    ax.set_title("predictor-target cosine over training")
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)

# (c) probe accuracy over time
if history.get("probe_accs"):
    ax = axes[2]
    eps, accs = zip(*history["probe_accs"])
    ax.plot(eps, [a*100 for a in accs], 'o-', color='C3', lw=1.5, markersize=6)
    ax.set_xlabel("epoch"); ax.set_ylabel("probe acc (%)")
    ax.set_ylim(0, 100)
    ax.set_title(f"linear probe accuracy (best = {max(accs)*100:.2f}%)")
    ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 9. (Optional) collapse diagnostics over time

In [ ]:
if history["diag_log"]:
    keys = ["loss", "|g_ctx|", "std_b(hp)", "std_t(g)", "|h_pred|", "|center|"]
    labels = ["JEPA loss", "‖g_ctx‖", "σ_batch(h_pred)  [collapse↓]",
              "σ_token(g_ctx)  [collapse↓]", "‖h_pred‖", "‖center‖"]
    fig, axes = plt.subplots(2, 3, figsize=(16, 7))
    steps = history["diag_steps"]
    for ax, k, lab in zip(axes.flat, keys, labels):
        vals = [d[k] for d in history["diag_log"]]
        ax.plot(steps, vals, lw=1.0)
        ax.set_title(lab); ax.set_xlabel("step"); ax.grid(alpha=0.3)
        if k == "loss": ax.set_yscale("log")
    plt.tight_layout(); plt.show()
else:
    print("No diagnostic log yet — run the training cell first.")


## 10. Final linear probe — full evaluation

A longer, more reliable probe on whatever encoder weights are currently loaded
(after training, or after resuming from a checkpoint). Uses more probe epochs
than the embedded 3-epoch checks so the number is less noisy.

  - `LOAD_BEST=True`   — load `pbb_best.pt` first (the best-loss checkpoint).
  - `LOAD_BEST=False`  — probe the encoder currently in memory.
  - `PROBE_EPOCHS=10`  — bump up for stability.

The probe trains a fresh `LinearProbe` on `train_eval_loader` (K_HALF context,
matches the test contract), then reports test accuracy on `test_loader`.
JEPA weights are untouched (`quick_probe` saves & restores `requires_grad`
and `training` flags).


In [ ]:
# ---- knobs ----
LOAD_BEST    = True         # set False to probe the encoder currently in memory
PROBE_EPOCHS = 10           # 10 = stable estimate; bump higher if you want
PROBE_LR     = 1e-3

if LOAD_BEST and os.path.exists(PBB_BEST):
    s = torch.load(PBB_BEST, map_location=DEVICE, weights_only=False)
    context_encoder.load_state_dict(s["context_encoder"])
    print(f"loaded BEST encoder weights from {PBB_BEST}  "
          f"(epoch {s.get('epoch', '?')},  best_loss={s.get('best_loss', float('nan')):.4f})")
elif LOAD_BEST:
    print(f"{PBB_BEST} not found — probing the encoder currently in memory")
else:
    print("probing the encoder currently in memory")

print(f"running probe ({PROBE_EPOCHS} epochs, lr={PROBE_LR}) on K_HALF context ...")
t0 = time.time()
final_acc = probe_now(num_epochs=PROBE_EPOCHS, lr=PROBE_LR)
dt = time.time() - t0
print(f"\nfinal probe acc(K_HALF) = {final_acc*100:.2f}%   ({dt:.1f}s)")

# Compare against the during-training probe trace if available
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    print(f"\nduring-training probe trace ({len(accs)} checkpoints):")
    for e, a in history["probe_accs"]:
        marker = "  <-- best so far" if a == max(accs) else ""
        print(f"  ep {e:3d}:  {a*100:5.2f}%{marker}")
    print(f"\nfinal probe vs best in-training probe:  "
          f"{final_acc*100:.2f}%  vs  {max(accs)*100:.2f}%  "
          f"(delta = {(final_acc - max(accs))*100:+.2f} pp)")

# Optional: append the final probe to history and re-save the checkpoint
APPEND_TO_HISTORY = True
if APPEND_TO_HISTORY:
    last_epoch = history["loss"].__len__() if history.get("loss") else 0
    history.setdefault("probe_accs", []).append((last_epoch, final_acc))
    if os.path.exists(PBB_LAST):
        s = torch.load(PBB_LAST, map_location=DEVICE, weights_only=False)
        s["history"] = history
        save_atomic(s, PBB_LAST)
        print(f"\nappended final probe to {PBB_LAST}'s history")
